In [1]:
import sys
from pathlib import Path

for base in (Path.cwd(), Path.cwd().parent):
    src = base / "src"
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))


# Driftalytics Quickstart
Tabular drift monitoring in a few minutes.

In [2]:
import numpy as np
import pandas as pd
from driftalytics import SlidingWindowRunner

rng = np.random.default_rng(42)
reference = pd.DataFrame({"a": rng.normal(size=1000), "b": rng.normal(size=1000)})
live = pd.DataFrame({"a": rng.normal(loc=0.8, size=300), "b": rng.normal(size=300)})

runner = SlidingWindowRunner(
    reference=reference,
    monitors=["psi", "ks", "missing_rate"],
    window_size=300,
)
report = runner.run(live)
print(report.summary())

Driftalytics report: 300 rows, 300 analyzed, 0 skipped, 2 alert(s), 1 window(s)
  psi: score=0.7202 threshold=0.2000 alert=True status=ok
  ks: score=0.3700 threshold=0.0500 alert=True status=ok
  missing_rate: score=0.0000 threshold=0.0500 alert=False status=ok


In [3]:
report.to_html("drift_report.html")
report.to_json("drift_report.json")
print("wrote drift_report.html and drift_report.json")

wrote drift_report.html and drift_report.json


Sliding mode scores each window. `include_partial=True` keeps a trailing short window.

In [4]:
windowed = SlidingWindowRunner(
    reference=reference,
    monitors=["psi", "missing_rate"],
    window_size=100,
    stride=100,
    sliding=True,
    include_partial=True,
)
window_report = windowed.run(live)
print(
    f"windows={window_report.n_windows} "
    f"analyzed={window_report.n_rows_analyzed} "
    f"skipped={window_report.n_rows_skipped}"
)

windows=3 analyzed=300 skipped=0
